# Chat Format and SFT


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import copy
from course_tools import DEFAULT_CHAT_MESSAGES, build_demo_bundle, evaluate_model, format_messages, train_model

LECTURE_NOTE_TITLE = 'Chat Format and SFT'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Chat Format and SFT


## Demo A: a conversation is flattened into one causal token stream


In [2]:
formatted = format_messages(DEFAULT_CHAT_MESSAGES + [{'role': 'assistant', 'content': 'Self-attention lets tokens mix information from relevant context.'}])
print(formatted)


<|system|>
You are a compact teaching assistant for an LLM course.
<|user|>
Explain self-attention in three short sentences.
<|assistant|>
Self-attention lets tokens mix information from relevant context.



## Demo B: base model vs SFT model


In [3]:
bundle = build_demo_bundle(steps=20)
tokenizer = bundle['tokenizer']
target_answer = 'Self-attention creates a weighted summary of earlier tokens.'
prompt = format_messages(DEFAULT_CHAT_MESSAGES, add_assistant_prompt=True)
target_chat = format_messages(
    DEFAULT_CHAT_MESSAGES + [{'role': 'assistant', 'content': target_answer}]
)
base_metrics = evaluate_model(bundle['model'], tokenizer, text='\n'.join([target_chat] * 8), batch_size=4)
print('base eval on chat-style target:', base_metrics)

sft_model = copy.deepcopy(bundle['model'])
sft_text = '\n'.join([target_chat] * 24)
history = train_model(sft_model, tokenizer, train_text=sft_text, eval_text=sft_text, steps=30, learning_rate=2e-3, batch_size=4)
print('sft history:', history)
sft_metrics = evaluate_model(sft_model, tokenizer, text='\n'.join([target_chat] * 8), batch_size=4)
print('sft eval on same target:', sft_metrics)


base eval on chat-style target: {'loss': 6.245710372924805, 'bpb': 9.010655381847375}
sft history: [{'step': 1.0, 'train_loss': 5.743241310119629, 'eval_loss': 5.198031902313232, 'bpb': 7.499174847849925}, {'step': 6.0, 'train_loss': 5.3510236740112305, 'eval_loss': 4.945338726043701, 'bpb': 7.134615655579392}, {'step': 12.0, 'train_loss': 4.138543605804443, 'eval_loss': 3.879585027694702, 'bpb': 5.597058080162219}, {'step': 18.0, 'train_loss': 3.5885887145996094, 'eval_loss': 3.314464569091797, 'bpb': 4.78176159703091}, {'step': 24.0, 'train_loss': 3.1823980808258057, 'eval_loss': 3.034261703491211, 'bpb': 4.377514312386069}, {'step': 30.0, 'train_loss': 2.7656161785125732, 'eval_loss': 2.847759485244751, 'bpb': 4.108448487007109}]
sft eval on same target: {'loss': 2.878204584121704, 'bpb': 4.152371480176264}


## Demo C: the architecture stayed the same, the training distribution changed


In [4]:
print(type(bundle['model']).__name__)
print(type(sft_model).__name__)
print('same architecture, different parameters after extra chat-style training')


TinyTransformerLM
TinyTransformerLM
same architecture, different parameters after extra chat-style training


## Rasbt and nanochat


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/gpt_instruction_finetuning.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/chat_sft.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/tokenizer.py
